In [10]:
import os

print(os.getcwd())
print(os.listdir())

/Users/saraelbachouti/Desktop/UCV-Churn
['eda5.ipynb', 'primermodelosimple.ipynb', '.DS_Store', 'LICENSE', 'requirements.txt', 'tercermodelo.ipynb', 'prueba_agentes', 'modelomasexacto.ipynb', 'config', 'Dockerfile', 'eda4.ipynb', 'edatercerenfoque.ipynb', 'models', 'docs', 'README.md', 'cuartomodelo.ipynb', '.gitignore', 'edasegundoenfoque.ipynb', 'UCV-Churn', 'dataset_limpio_churn.csv', '.git', 'processed', 'data', 'edaprimerenfoque.ipynb', 'notebooks', 'reports', 'src']


In [12]:
# ============================================================
# MODELO FINAL XGBOOST CORRECTO
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

from xgboost import XGBClassifier


# ============================================================
# 1. CARGAR DATASET
# ============================================================

df_model = pd.read_csv("dataset_limpio_churn.csv")

dataset = df_model.copy()

print("Shape inicial:", dataset.shape)


# ============================================================
# 2. FECHAS
# ============================================================

if "fecha" in dataset.columns:

    dataset["fecha"] = pd.to_datetime(
        dataset["fecha"],
        errors="coerce"
    )

    if "cliente_id" in dataset.columns:

        dataset = dataset.sort_values(
            ["cliente_id", "fecha"]
        )


# ============================================================
# 3. FEATURES TEMPORALES
# ============================================================

if (
    "cliente_id" in dataset.columns
    and
    "importe_total" in dataset.columns
):

    dataset["media_importe_3m"] = (
        dataset
        .groupby("cliente_id")["importe_total"]
        .transform(
            lambda x:
            x.rolling(
                3,
                min_periods=1
            ).mean()
        )
    )

    dataset["importe_mes_anterior"] = (
        dataset
        .groupby("cliente_id")
        ["importe_total"]
        .shift(1)
    )

    dataset["subida_factura"] = (
        dataset["importe_total"]
        -
        dataset["importe_mes_anterior"]
    )


if (
    "cliente_id" in dataset.columns
    and
    "dias_retraso_pago" in dataset.columns
):

    dataset["media_retraso_3m"] = (

        dataset

        .groupby("cliente_id")

        ["dias_retraso_pago"]

        .transform(
            lambda x:
            x.rolling(
                3,
                min_periods=1
            ).mean()
        )
    )


if (
    "cliente_id" in dataset.columns
    and
    "impago_flag" in dataset.columns
):

    dataset["impagos_3m"] = (

        dataset

        .groupby("cliente_id")

        ["impago_flag"]

        .transform(
            lambda x:
            x.rolling(
                3,
                min_periods=1
            ).sum()
        )
    )


# ============================================================
# 4. VARIABLES INTERACCIÓN
# ============================================================

if (
    "dias_retraso_pago" in dataset.columns
    and
    "importe_total" in dataset.columns
):

    dataset["riesgo_pago"] = (
        dataset["dias_retraso_pago"]
        *
        dataset["importe_total"]
    )


if (
    "impago_flag" in dataset.columns
    and
    "stress_calidad_lag" in dataset.columns
):

    dataset["riesgo_finanzas"] = (
        dataset["impago_flag"]
        *
        dataset["stress_calidad_lag"]
    )


# ============================================================
# 5. ELIMINAR COLUMNAS
# ============================================================

eliminar = [

    "cliente_id",
    "fecha",
    "zona_id",
    "zona_id_x",
    "zona_id_y"

]

dataset = dataset.drop(
    columns=[
        c
        for c
        in eliminar
        if c
        in dataset.columns
    ],
    errors="ignore"
)


# ============================================================
# 6. TARGET
# ============================================================

if "churn" not in dataset.columns:

    raise Exception(
        "No existe columna churn"
    )


X = dataset.drop(
    columns="churn"
)

y = dataset["churn"]


# ============================================================
# 7. VARIABLES CATEGÓRICAS
# ============================================================

X = pd.get_dummies(
    X,
    drop_first=True
)


# ============================================================
# 8. LIMPIEZA
# ============================================================

X = X.replace(
    [np.inf, -np.inf],
    np.nan
)

X = X.fillna(
    X.median(
        numeric_only=True
    )
)

X = X.fillna(0)


# ============================================================
# 9. TRAIN TEST
# ============================================================

X_train, X_test, y_train, y_test = (

    train_test_split(

        X,
        y,

        test_size=0.20,

        random_state=42,

        stratify=y

    )

)


# ============================================================
# 10. BALANCEO
# ============================================================

peso = (
    (y_train == 0).sum()
    /
    (y_train == 1).sum()
)

print(
    "scale_pos_weight:",
    round(peso,2)
)


# ============================================================
# 11. MODELO
# ============================================================

modelo = XGBClassifier(

    n_estimators=500,

    learning_rate=0.05,

    max_depth=4,

    min_child_weight=10,

    subsample=0.8,

    colsample_bytree=0.8,

    scale_pos_weight=peso,

    random_state=42,

    eval_metric="logloss",

    n_jobs=-1

)


modelo.fit(
    X_train,
    y_train
)


# ============================================================
# 12. PREDICCIÓN
# ============================================================

proba = (
    modelo
    .predict_proba(
        X_test
    )[:,1]
)


threshold = 0.60


pred = (
    proba
    >=
    threshold
).astype(int)


# ============================================================
# RESULTADOS
# ============================================================

print("\nRESULTADOS")

print(
"Accuracy:",
round(
accuracy_score(
y_test,
pred
)*100,
2
)
)

print(
"Precision:",
round(
precision_score(
y_test,
pred,
zero_division=0
),
4
)
)

print(
"Recall:",
round(
recall_score(
y_test,
pred
),
4
)
)

print(
"F1:",
round(
f1_score(
y_test,
pred
),
4
)
)

print(
"ROC:",
round(
roc_auc_score(
y_test,
proba
),
4
)
)

print(
"PR:",
round(
average_precision_score(
y_test,
proba
),
4
)
)

print(
"\nMatriz:"
)

print(
confusion_matrix(
y_test,
pred
)
)

print(
"\nClassification Report"
)

print(
classification_report(
y_test,
pred
)
)

Shape inicial: (317579, 35)
scale_pos_weight: 160.41

RESULTADOS
Accuracy: 90.26
Precision: 0.0213
Recall: 0.3274
F1: 0.04
ROC: 0.7078
PR: 0.0237

Matriz:
[[57200  5922]
 [  265   129]]

Classification Report
              precision    recall  f1-score   support

           0       1.00      0.91      0.95     63122
           1       0.02      0.33      0.04       394

    accuracy                           0.90     63516
   macro avg       0.51      0.62      0.49     63516
weighted avg       0.99      0.90      0.94     63516



Aunque el modelo con una exactitud del 90,26 % presentó un mayor porcentaje de aciertos globales, no fue seleccionado como modelo final debido a que su capacidad para detectar clientes con riesgo de abandono era inferior. En concreto, identificó 129 clientes churn reales, frente a los 148 detectados por la configuración finalmente elegida. Dado que el objetivo principal del estudio es anticipar el abandono y maximizar la detección de clientes con riesgo de churn, se priorizaron métricas como el recall, el F1-score y la capacidad discriminativa del modelo frente a la exactitud global. Por ello, se optó por la configuración con un umbral de decisión de 0,60, que permitió alcanzar un recall del 37,56 % y detectar un mayor número de clientes potencialmente susceptibles de abandonar la compañía, aun a costa de una ligera reducción en la accuracy.


In [7]:
# ============================================================
# MODELO FINAL TFM — XGBOOST CHURN DEFENDIBLE
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

from xgboost import XGBClassifier


# ======================================================
# 1. CARGAR DATASET
# ======================================================

df = pd.read_csv("dataset_limpio_churn.csv")

print("Shape inicial:", df.shape)


# ======================================================
# 2. CREAR VARIABLES DERIVADAS
# ======================================================

df["cliente_moroso"] = np.where(
    df["dias_retraso_pago"] > 0,
    1,
    0
)

df["riesgo_financiero"] = (
    df["impago_flag"].fillna(0) * 5
    + df["dias_retraso_pago"].fillna(0)
)

df["riesgo_consumo"] = (
    df["variacion_consumo_pct"].fillna(0).abs()
)

df["retraso_impago"] = (
    df["dias_retraso_pago"].fillna(0)
    * df["impago_flag"].fillna(0)
)


# ======================================================
# 3. ELIMINAR VARIABLES NO VÁLIDAS
# ======================================================

quitar = [
    "cliente_id",
    "fecha",
    "alerta_churn",
    "zona_id",
    "zona_id_x",
    "zona_id_y"
]

df = df.drop(
    columns=[c for c in quitar if c in df.columns],
    errors="ignore"
)


# ======================================================
# 4. TARGET
# ======================================================

X = df.drop(columns="churn")
y = df["churn"]


# ======================================================
# 5. VARIABLES CATEGÓRICAS
# ======================================================

X = pd.get_dummies(
    X,
    drop_first=True
)


# ======================================================
# 6. LIMPIEZA
# ======================================================

X = X.replace(
    [np.inf, -np.inf],
    np.nan
)

X = X.fillna(
    X.median(numeric_only=True)
)

X = X.fillna(0)


# ======================================================
# 7. TRAIN / TEST
# ======================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    stratify=y,
    test_size=0.20,
    random_state=42
)


# ======================================================
# 8. PESO CLASE MINORITARIA
# ======================================================

peso = (y_train == 0).sum() / (y_train == 1).sum()

print("\nscale_pos_weight:", round(peso, 2))


# ======================================================
# 9. MODELO XGBOOST
# ======================================================

modelo = XGBClassifier(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=3,
    min_child_weight=20,
    gamma=5,
    subsample=0.80,
    colsample_bytree=0.80,
    reg_alpha=5,
    reg_lambda=10,
    scale_pos_weight=peso,
    objective="binary:logistic",
    eval_metric="aucpr",
    random_state=42,
    n_jobs=-1
)

modelo.fit(X_train, y_train)


# ======================================================
# 10. PROBABILIDADES
# ======================================================

proba = modelo.predict_proba(X_test)[:, 1]


# ======================================================
# 11. TABLA DE THRESHOLDS
# ======================================================

resultados = []

for threshold in np.arange(0.30, 0.81, 0.01):

    pred_temp = (proba >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        pred_temp
    ).ravel()

    resultados.append({
        "threshold": round(threshold, 2),
        "accuracy": accuracy_score(y_test, pred_temp),
        "precision": precision_score(y_test, pred_temp, zero_division=0),
        "recall": recall_score(y_test, pred_temp, zero_division=0),
        "f1": f1_score(y_test, pred_temp, zero_division=0),
        "clientes_churn_detectados": tp,
        "clientes_churn_no_detectados": fn,
        "falsos_positivos": fp,
        "total_predichos_churn": tp + fp
    })

df_thresholds = pd.DataFrame(resultados)


print("\n" + "="*70)
print("TABLA DE THRESHOLDS ORDENADA POR F1")
print("="*70)

print(
    df_thresholds
    .sort_values(by="f1", ascending=False)
    .head(15)
)


print("\n" + "="*70)
print("TABLA DE THRESHOLDS PARA DETECTAR MÁS CHURN")
print("="*70)

print(
    df_thresholds
    .sort_values(by="clientes_churn_detectados", ascending=False)
    .head(15)
)


# ======================================================
# 12. ELEGIR THRESHOLD FINAL
# ======================================================
# Puedes cambiar este valor:
# 0.70 = más equilibrado
# 0.60 = detecta más churn, pero aumenta falsos positivos
# 0.55 = detecta todavía más, pero con más ruido

threshold_final = 0.60

pred = (proba >= threshold_final).astype(int)


# ======================================================
# 13. RESULTADOS FINALES
# ======================================================

tn, fp, fn, tp = confusion_matrix(
    y_test,
    pred
).ravel()

accuracy = accuracy_score(y_test, pred)
precision = precision_score(y_test, pred, zero_division=0)
recall = recall_score(y_test, pred, zero_division=0)
f1 = f1_score(y_test, pred, zero_division=0)
roc = roc_auc_score(y_test, proba)
pr_auc = average_precision_score(y_test, proba)


print("\n" + "="*70)
print("RESULTADO FINAL XGBOOST")
print("="*70)

print("Threshold utilizado:", threshold_final)
print("Accuracy:", round(accuracy * 100, 2))
print("Precision churn:", round(precision, 4))
print("Recall churn:", round(recall, 4))
print("F1 churn:", round(f1, 4))
print("ROC-AUC:", round(roc, 4))
print("PR-AUC:", round(pr_auc, 4))

print("\nMatriz de confusión:")
print(confusion_matrix(y_test, pred))

print("\nClientes churn reales:", tp + fn)
print("Clientes churn detectados:", tp)
print("Clientes churn no detectados:", fn)
print("Falsos positivos:", fp)
print("Total predichos como churn:", tp + fp)

print("\nClassification Report:")
print(classification_report(y_test, pred, zero_division=0))


# ======================================================
# 14. IMPORTANCIA DE VARIABLES
# ======================================================

importancias = pd.DataFrame({
    "Variable": X.columns,
    "Importancia": modelo.feature_importances_
}).sort_values(
    by="Importancia",
    ascending=False
)

print("\n" + "="*70)
print("TOP 20 VARIABLES MÁS IMPORTANTES")
print("="*70)

print(importancias.head(20))

Shape inicial: (317579, 35)

scale_pos_weight: 160.41

TABLA DE THRESHOLDS ORDENADA POR F1
    threshold  accuracy  precision    recall        f1  \
46       0.76  0.966355   0.031200  0.147208  0.051487   
45       0.75  0.961868   0.028372  0.154822  0.047956   
49       0.79  0.976352   0.031303  0.093909  0.046954   
48       0.78  0.973125   0.030064  0.106599  0.046901   
47       0.77  0.969882   0.029138  0.119289  0.046836   
44       0.74  0.957381   0.026994  0.167513  0.046495   
43       0.73  0.952658   0.026459  0.185279  0.046305   
50       0.80  0.979123   0.032129  0.081218  0.046043   
42       0.72  0.948234   0.025262  0.195431  0.044741   
41       0.71  0.942991   0.024462  0.210660  0.043834   
40       0.70  0.937858   0.023854  0.225888  0.043152   
39       0.69  0.932836   0.023387  0.241117  0.042639   
38       0.68  0.926680   0.022192  0.251269  0.040783   
36       0.66  0.914982   0.021232  0.281726  0.039488   
37       0.67  0.920603   0.021215  0.2

In [4]:
# ============================================================
# MODELO FINAL TFM — XGBOOST CHURN MÁS SENSIBLE
# churn_t_plus_1 + split temporal + threshold por recall mínimo
# ============================================================

import pandas as pd
import numpy as np

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    fbeta_score, roc_auc_score, average_precision_score,
    confusion_matrix, classification_report
)

from xgboost import XGBClassifier


# ======================================================
# 1. CARGAR DATASET
# ======================================================

df = pd.read_csv("dataset_limpio_churn.csv")

print("Shape inicial:", df.shape)

df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")
df = df.sort_values(["cliente_id", "fecha"])


# ======================================================
# 2. CREAR TARGET FUTURO
# ======================================================

df["churn_t_plus_1"] = (
    df.groupby("cliente_id")["churn"]
    .shift(-1)
)

df = df.dropna(subset=["churn_t_plus_1"])
df["churn_t_plus_1"] = df["churn_t_plus_1"].astype(int)


# ======================================================
# 3. VARIABLES DERIVADAS
# ======================================================

df["cliente_moroso"] = np.where(df["dias_retraso_pago"] > 0, 1, 0)

df["riesgo_financiero"] = (
    df["impago_flag"].fillna(0) * 5
    + df["dias_retraso_pago"].fillna(0)
)

df["riesgo_consumo"] = df["variacion_consumo_pct"].fillna(0).abs()

df["retraso_impago"] = (
    df["dias_retraso_pago"].fillna(0)
    * df["impago_flag"].fillna(0)
)

df["subida_factura_flag"] = np.where(
    df["subida_factura"].fillna(0) > 0,
    1,
    0
)

df["bajada_consumo_flag"] = np.where(
    df["variacion_consumo_pct"].fillna(0) < 0,
    1,
    0
)


# ======================================================
# 4. ELIMINAR VARIABLES CON LEAKAGE O NO VÁLIDAS
# ======================================================

quitar = [
    "cliente_id",
    "fecha",
    "churn",
    "alerta_churn",
    "zona_id",
    "zona_id_x",
    "zona_id_y"
]

df_model = df.drop(
    columns=[c for c in quitar if c in df.columns],
    errors="ignore"
)


# ======================================================
# 5. FEATURES Y TARGET
# ======================================================

X = df_model.drop(columns="churn_t_plus_1")
y = df_model["churn_t_plus_1"]

X = pd.get_dummies(X, drop_first=True)

X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median(numeric_only=True))
X = X.fillna(0)


# ======================================================
# 6. SPLIT TEMPORAL TRAIN / VALID / TEST
# ======================================================

fechas = df.loc[X.index, "fecha"]

datos = X.copy()
datos["target"] = y.values
datos["fecha"] = fechas.values

datos = datos.sort_values("fecha")

n = len(datos)

train_end = int(n * 0.70)
valid_end = int(n * 0.85)

train = datos.iloc[:train_end]
valid = datos.iloc[train_end:valid_end]
test = datos.iloc[valid_end:]

X_train = train.drop(columns=["target", "fecha"])
y_train = train["target"]

X_valid = valid.drop(columns=["target", "fecha"])
y_valid = valid["target"]

X_test = test.drop(columns=["target", "fecha"])
y_test = test["target"]

print("\nTamaños:")
print("Train:", X_train.shape)
print("Valid:", X_valid.shape)
print("Test:", X_test.shape)

print("\nChurn rate:")
print("Train:", round(y_train.mean() * 100, 4), "%")
print("Valid:", round(y_valid.mean() * 100, 4), "%")
print("Test:", round(y_test.mean() * 100, 4), "%")


# ======================================================
# 7. PESO CLASE MINORITARIA
# ======================================================

peso = (y_train == 0).sum() / (y_train == 1).sum()

print("\nscale_pos_weight:", round(peso, 2))


# ======================================================
# 8. MODELO XGBOOST MÁS SENSIBLE
# ======================================================

modelo = XGBClassifier(
    n_estimators=900,
    learning_rate=0.025,
    max_depth=3,
    min_child_weight=8,
    gamma=1,
    subsample=0.90,
    colsample_bytree=0.90,
    reg_alpha=1,
    reg_lambda=4,
    scale_pos_weight=peso * 1.5,
    objective="binary:logistic",
    eval_metric="aucpr",
    random_state=42,
    n_jobs=-1
)

modelo.fit(
    X_train,
    y_train,
    eval_set=[(X_valid, y_valid)],
    verbose=False
)


# ======================================================
# 9. PROBABILIDADES VALIDACIÓN
# ======================================================

proba_valid = modelo.predict_proba(X_valid)[:, 1]


# ======================================================
# 10. TABLA DE THRESHOLDS EN VALIDACIÓN
# ======================================================

resultados = []

for threshold in np.arange(0.20, 0.81, 0.01):

    pred_valid = (proba_valid >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_valid,
        pred_valid
    ).ravel()

    resultados.append({
        "threshold": round(threshold, 2),
        "accuracy": accuracy_score(y_valid, pred_valid),
        "precision": precision_score(y_valid, pred_valid, zero_division=0),
        "recall": recall_score(y_valid, pred_valid, zero_division=0),
        "f1": f1_score(y_valid, pred_valid, zero_division=0),
        "f2": fbeta_score(y_valid, pred_valid, beta=2, zero_division=0),
        "clientes_churn_detectados": tp,
        "clientes_churn_no_detectados": fn,
        "falsos_positivos": fp,
        "total_predichos_churn": tp + fp
    })

df_thresholds = pd.DataFrame(resultados)

print("\n" + "="*70)
print("THRESHOLDS ORDENADOS POR RECALL")
print("="*70)

print(
    df_thresholds
    .sort_values(by="recall", ascending=False)
    .head(20)
)

print("\n" + "="*70)
print("THRESHOLDS ORDENADOS POR F2")
print("="*70)

print(
    df_thresholds
    .sort_values(by="f2", ascending=False)
    .head(20)
)


# ======================================================
# 11. ELEGIR THRESHOLD FINAL
# ======================================================
# Objetivo: detectar más churn.
# Se exige mínimo 20% de recall en validación.
# Si no hay ningún threshold con recall >= 0.20, usa el mejor F2.

recall_minimo = 0.20

candidatos = df_thresholds[df_thresholds["recall"] >= recall_minimo]

if len(candidatos) > 0:
    threshold_final = (
        candidatos
        .sort_values(by="f2", ascending=False)
        .iloc[0]["threshold"]
    )
else:
    threshold_final = (
        df_thresholds
        .sort_values(by="f2", ascending=False)
        .iloc[0]["threshold"]
    )

print("\nThreshold final elegido:", threshold_final)


# ======================================================
# 12. EVALUACIÓN FINAL EN TEST
# ======================================================

proba_test = modelo.predict_proba(X_test)[:, 1]

pred_test = (proba_test >= threshold_final).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_test,
    pred_test
).ravel()

accuracy = accuracy_score(y_test, pred_test)
precision = precision_score(y_test, pred_test, zero_division=0)
recall = recall_score(y_test, pred_test, zero_division=0)
f1 = f1_score(y_test, pred_test, zero_division=0)
f2 = fbeta_score(y_test, pred_test, beta=2, zero_division=0)
roc = roc_auc_score(y_test, proba_test)
pr_auc = average_precision_score(y_test, proba_test)

print("\n" + "="*70)
print("RESULTADO FINAL XGBOOST — TEST TEMPORAL")
print("="*70)

print("Threshold utilizado:", threshold_final)
print("Accuracy:", round(accuracy * 100, 2))
print("Precision churn:", round(precision, 4))
print("Recall churn:", round(recall, 4))
print("F1 churn:", round(f1, 4))
print("F2 churn:", round(f2, 4))
print("ROC-AUC:", round(roc, 4))
print("PR-AUC:", round(pr_auc, 4))

print("\nMatriz de confusión:")
print(confusion_matrix(y_test, pred_test))

print("\nClientes churn reales:", tp + fn)
print("Clientes churn detectados:", tp)
print("Clientes churn no detectados:", fn)
print("Falsos positivos:", fp)
print("Total predichos como churn:", tp + fp)

print("\nClassification Report:")
print(classification_report(y_test, pred_test, zero_division=0))


# ======================================================
# 13. IMPORTANCIA DE VARIABLES
# ======================================================

importancias = pd.DataFrame({
    "Variable": X_train.columns,
    "Importancia": modelo.feature_importances_
}).sort_values(
    by="Importancia",
    ascending=False
)

print("\n" + "="*70)
print("TOP 20 VARIABLES MÁS IMPORTANTES")
print("="*70)

print(importancias.head(20))

Shape inicial: (317579, 35)

Tamaños:
Train: (215305, 49)
Valid: (46137, 49)
Test: (46137, 49)

Churn rate:
Train: 0.6818 %
Valid: 0.531 %
Test: 0.5137 %

scale_pos_weight: 145.67

THRESHOLDS ORDENADOS POR RECALL
    threshold  accuracy  precision    recall        f1        f2  \
0        0.20  0.238160   0.006029  0.869388  0.011975  0.029331   
1        0.21  0.255435   0.006054  0.853061  0.012022  0.029432   
2        0.22  0.272081   0.006074  0.836735  0.012061  0.029514   
3        0.23  0.288857   0.006127  0.824490  0.012164  0.029751   
4        0.24  0.305698   0.006213  0.816327  0.012333  0.030150   
5        0.25  0.323168   0.006247  0.800000  0.012398  0.030290   
6        0.26  0.341093   0.006319  0.787755  0.012538  0.030615   
7        0.27  0.359039   0.006462  0.783673  0.012819  0.031280   
8        0.28  0.376682   0.006576  0.775510  0.013041  0.031801   
9        0.29  0.394239   0.006695  0.767347  0.013275  0.032348   
10       0.30  0.411839   0.006822  0.7

In [5]:
# ============================================================
# MODELO FINAL TFM — XGBOOST CHURN RECALL ALTO
# churn_t_plus_1 + split temporal + threshold por F2/recall
# ============================================================

import pandas as pd
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

from xgboost import XGBClassifier


# ======================================================
# 1. CARGAR DATASET
# ======================================================

df = pd.read_csv("dataset_limpio_churn.csv")

print("Shape inicial:", df.shape)


# ======================================================
# 2. FECHA Y ORDEN TEMPORAL
# ======================================================

df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")

df = df.sort_values(["cliente_id", "fecha"])


# ======================================================
# 3. CREAR TARGET FUTURO
# ======================================================

df["churn_t_plus_1"] = (
    df.groupby("cliente_id")["churn"]
    .shift(-1)
)

df = df.dropna(subset=["churn_t_plus_1"])

df["churn_t_plus_1"] = df["churn_t_plus_1"].astype(int)


# ======================================================
# 4. VARIABLES DERIVADAS
# ======================================================

df["cliente_moroso"] = np.where(
    df["dias_retraso_pago"] > 0,
    1,
    0
)

df["riesgo_financiero"] = (
    df["impago_flag"].fillna(0) * 5
    + df["dias_retraso_pago"].fillna(0)
)

df["riesgo_consumo"] = (
    df["variacion_consumo_pct"].fillna(0).abs()
)

df["retraso_impago"] = (
    df["dias_retraso_pago"].fillna(0)
    * df["impago_flag"].fillna(0)
)

df["subida_factura_flag"] = np.where(
    df["subida_factura"].fillna(0) > 0,
    1,
    0
)

df["bajada_consumo_flag"] = np.where(
    df["variacion_consumo_pct"].fillna(0) < 0,
    1,
    0
)


# ======================================================
# 5. ELIMINAR VARIABLES NO VÁLIDAS / LEAKAGE
# ======================================================

quitar = [
    "cliente_id",
    "fecha",
    "churn",
    "alerta_churn",
    "zona_id",
    "zona_id_x",
    "zona_id_y"
]

df_model = df.drop(
    columns=[c for c in quitar if c in df.columns],
    errors="ignore"
)


# ======================================================
# 6. FEATURES Y TARGET
# ======================================================

X = df_model.drop(columns="churn_t_plus_1")
y = df_model["churn_t_plus_1"]


# ======================================================
# 7. VARIABLES CATEGÓRICAS
# ======================================================

X = pd.get_dummies(
    X,
    drop_first=True
)


# ======================================================
# 8. LIMPIEZA
# ======================================================

X = X.replace(
    [np.inf, -np.inf],
    np.nan
)

X = X.fillna(
    X.median(numeric_only=True)
)

X = X.fillna(0)


# ======================================================
# 9. SPLIT TEMPORAL TRAIN / VALID / TEST
# ======================================================

fechas = df.loc[X.index, "fecha"]

datos = X.copy()
datos["target"] = y.values
datos["fecha"] = fechas.values

datos = datos.sort_values("fecha")

n = len(datos)

train_end = int(n * 0.70)
valid_end = int(n * 0.85)

train = datos.iloc[:train_end]
valid = datos.iloc[train_end:valid_end]
test = datos.iloc[valid_end:]

X_train = train.drop(columns=["target", "fecha"])
y_train = train["target"]

X_valid = valid.drop(columns=["target", "fecha"])
y_valid = valid["target"]

X_test = test.drop(columns=["target", "fecha"])
y_test = test["target"]

print("\nTamaños:")
print("Train:", X_train.shape)
print("Valid:", X_valid.shape)
print("Test:", X_test.shape)

print("\nChurn rate:")
print("Train:", round(y_train.mean() * 100, 4), "%")
print("Valid:", round(y_valid.mean() * 100, 4), "%")
print("Test:", round(y_test.mean() * 100, 4), "%")


# ======================================================
# 10. PESO CLASE MINORITARIA
# ======================================================

peso = (y_train == 0).sum() / (y_train == 1).sum()

print("\nscale_pos_weight base:", round(peso, 2))
print("scale_pos_weight usado:", round(peso * 2.5, 2))


# ======================================================
# 11. MODELO XGBOOST RECALL ALTO
# ======================================================

modelo = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.02,
    max_depth=3,
    min_child_weight=5,
    gamma=0.5,
    subsample=0.90,
    colsample_bytree=0.90,
    reg_alpha=1,
    reg_lambda=3,
    scale_pos_weight=peso * 2.5,
    objective="binary:logistic",
    eval_metric="aucpr",
    random_state=42,
    n_jobs=-1
)

modelo.fit(
    X_train,
    y_train,
    eval_set=[(X_valid, y_valid)],
    verbose=False
)


# ======================================================
# 12. PROBABILIDADES VALIDACIÓN
# ======================================================

proba_valid = modelo.predict_proba(X_valid)[:, 1]


# ======================================================
# 13. TABLA DE THRESHOLDS EN VALIDACIÓN
# ======================================================

resultados = []

for threshold in np.arange(0.10, 0.81, 0.01):

    pred_valid = (proba_valid >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_valid,
        pred_valid
    ).ravel()

    resultados.append({
        "threshold": round(threshold, 2),
        "accuracy": accuracy_score(y_valid, pred_valid),
        "precision": precision_score(y_valid, pred_valid, zero_division=0),
        "recall": recall_score(y_valid, pred_valid, zero_division=0),
        "f1": f1_score(y_valid, pred_valid, zero_division=0),
        "f2": fbeta_score(y_valid, pred_valid, beta=2, zero_division=0),
        "clientes_churn_detectados": tp,
        "clientes_churn_no_detectados": fn,
        "falsos_positivos": fp,
        "total_predichos_churn": tp + fp
    })

df_thresholds = pd.DataFrame(resultados)

print("\n" + "="*70)
print("THRESHOLDS ORDENADOS POR RECALL")
print("="*70)

print(
    df_thresholds
    .sort_values(by="recall", ascending=False)
    .head(20)
)

print("\n" + "="*70)
print("THRESHOLDS ORDENADOS POR F2")
print("="*70)

print(
    df_thresholds
    .sort_values(by="f2", ascending=False)
    .head(20)
)


# ======================================================
# 14. ELEGIR THRESHOLD FINAL
# ======================================================
# Objetivo: detectar más churn.
# Pedimos al menos 35% de recall en validación.
# Si no se consigue, usa el threshold con mejor F2.

recall_minimo = 0.35

candidatos = df_thresholds[
    df_thresholds["recall"] >= recall_minimo
]

if len(candidatos) > 0:
    threshold_final = (
        candidatos
        .sort_values(by="f2", ascending=False)
        .iloc[0]["threshold"]
    )
else:
    threshold_final = (
        df_thresholds
        .sort_values(by="f2", ascending=False)
        .iloc[0]["threshold"]
    )

print("\nThreshold final elegido:", threshold_final)


# ======================================================
# 15. EVALUACIÓN FINAL EN TEST
# ======================================================

proba_test = modelo.predict_proba(X_test)[:, 1]

pred_test = (proba_test >= threshold_final).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_test,
    pred_test
).ravel()

accuracy = accuracy_score(y_test, pred_test)
precision = precision_score(y_test, pred_test, zero_division=0)
recall = recall_score(y_test, pred_test, zero_division=0)
f1 = f1_score(y_test, pred_test, zero_division=0)
f2 = fbeta_score(y_test, pred_test, beta=2, zero_division=0)
roc = roc_auc_score(y_test, proba_test)
pr_auc = average_precision_score(y_test, proba_test)


print("\n" + "="*70)
print("RESULTADO FINAL XGBOOST — TEST TEMPORAL")
print("="*70)

print("Threshold utilizado:", threshold_final)
print("Accuracy:", round(accuracy * 100, 2))
print("Precision churn:", round(precision, 4))
print("Recall churn:", round(recall, 4))
print("F1 churn:", round(f1, 4))
print("F2 churn:", round(f2, 4))
print("ROC-AUC:", round(roc, 4))
print("PR-AUC:", round(pr_auc, 4))

print("\nMatriz de confusión:")
print(confusion_matrix(y_test, pred_test))

print("\nClientes churn reales:", tp + fn)
print("Clientes churn detectados:", tp)
print("Clientes churn no detectados:", fn)
print("Falsos positivos:", fp)
print("Total predichos como churn:", tp + fp)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        pred_test,
        zero_division=0
    )
)


# ======================================================
# 16. IMPORTANCIA DE VARIABLES
# ======================================================

importancias = pd.DataFrame({
    "Variable": X_train.columns,
    "Importancia": modelo.feature_importances_
}).sort_values(
    by="Importancia",
    ascending=False
)

print("\n" + "="*70)
print("TOP 20 VARIABLES MÁS IMPORTANTES")
print("="*70)

print(importancias.head(20))

Shape inicial: (317579, 35)

Tamaños:
Train: (215305, 49)
Valid: (46137, 49)
Test: (46137, 49)

Churn rate:
Train: 0.6818 %
Valid: 0.531 %
Test: 0.5137 %

scale_pos_weight base: 145.67
scale_pos_weight usado: 364.16

THRESHOLDS ORDENADOS POR RECALL
    threshold  accuracy  precision    recall        f1        f2  \
0        0.10  0.053124   0.005419  0.971429  0.010778  0.026505   
1        0.11  0.060797   0.005418  0.963265  0.010775  0.026494   
2        0.12  0.069684   0.005446  0.959184  0.010831  0.026628   
3        0.13  0.078462   0.005498  0.959184  0.010934  0.026874   
4        0.14  0.087002   0.005503  0.951020  0.010942  0.026890   
5        0.15  0.096582   0.005561  0.951020  0.011057  0.027168   
6        0.16  0.105945   0.005571  0.942857  0.011076  0.027211   
7        0.17  0.116414   0.005564  0.930612  0.011062  0.027171   
8        0.18  0.126991   0.005558  0.918367  0.011049  0.027133   
9        0.19  0.137829   0.005627  0.918367  0.011186  0.027464   
10 

Tras comparar los distintos modelos desarrollados, se seleccionó finalmente el modelo XGBoost con un umbral de decisión de 0,69, ya que fue el que proporcionó el mejor equilibrio entre capacidad de detección del churn y rendimiento global del sistema.

Aunque algunos modelos alcanzaban una mayor exactitud, estos presentaban una menor capacidad para identificar a los clientes que realmente abandonaban el servicio. Dado que el objetivo principal del estudio era detectar el mayor número posible de clientes con riesgo de baja, se decidió priorizar la métrica recall frente a la accuracy.

El modelo seleccionado obtuvo una accuracy del 85,18% y un recall del 39,66%, siendo capaz de identificar 94 de los 237 clientes que abandonaron el servicio en el conjunto de prueba. Además, mantuvo unos niveles de rendimiento global satisfactorios y un comportamiento más equilibrado que otros modelos analizados.

Por todo ello, se considera que este modelo constituye la alternativa más adecuada para un sistema de detección temprana de churn, permitiendo a la empresa implementar estrategias de retención sobre aquellos clientes con mayor probabilidad de abandono.


In [7]:
# ============================================================
# MODELO FINAL TFM — XGBOOST CHURN DEFENDIBLE
# churn_t_plus_1 + split temporal + threshold por recall/F2
# ============================================================

import pandas as pd
import numpy as np

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    fbeta_score, roc_auc_score, average_precision_score,
    confusion_matrix, classification_report
)

from xgboost import XGBClassifier


# ======================================================
# 1. CARGAR DATASET
# ======================================================

df = pd.read_csv("dataset_limpio_churn.csv")

print("Shape inicial:", df.shape)
print("Columnas:", df.columns.tolist())

df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")
df = df.sort_values(["cliente_id", "fecha"])


# ======================================================
# 2. CREAR TARGET FUTURO
# ======================================================

df["churn_t_plus_1"] = (
    df.groupby("cliente_id")["churn"]
    .shift(-1)
)

df = df.dropna(subset=["churn_t_plus_1"])
df["churn_t_plus_1"] = df["churn_t_plus_1"].astype(int)


# ======================================================
# 3. VARIABLES HISTÓRICAS SIN LEAKAGE
# ======================================================

df["importe_lag1"] = df.groupby("cliente_id")["importe_total"].shift(1)
df["retraso_lag1"] = df.groupby("cliente_id")["dias_retraso_pago"].shift(1)
df["impago_lag1"] = df.groupby("cliente_id")["impago_flag"].shift(1)

df["media_importe_3m_lag"] = (
    df.groupby("cliente_id")["importe_total"]
    .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
)

df["media_retraso_3m_lag"] = (
    df.groupby("cliente_id")["dias_retraso_pago"]
    .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
)

df["impagos_3m_lag"] = (
    df.groupby("cliente_id")["impago_flag"]
    .transform(lambda x: x.shift(1).rolling(3, min_periods=1).sum())
)

df["media_importe_6m_lag"] = (
    df.groupby("cliente_id")["importe_total"]
    .transform(lambda x: x.shift(1).rolling(6, min_periods=1).mean())
)

df["media_retraso_6m_lag"] = (
    df.groupby("cliente_id")["dias_retraso_pago"]
    .transform(lambda x: x.shift(1).rolling(6, min_periods=1).mean())
)

df["impagos_6m_lag"] = (
    df.groupby("cliente_id")["impago_flag"]
    .transform(lambda x: x.shift(1).rolling(6, min_periods=1).sum())
)


# ======================================================
# 4. VARIABLES DERIVADAS
# ======================================================

df["cliente_moroso_lag"] = np.where(
    df["retraso_lag1"].fillna(0) > 0,
    1,
    0
)

df["riesgo_financiero_lag"] = (
    df["impago_lag1"].fillna(0) * 5
    + df["retraso_lag1"].fillna(0)
)

df["subida_factura_lag"] = (
    df["importe_lag1"] - df["media_importe_3m_lag"]
)

df["variacion_importe_lag"] = (
    df["subida_factura_lag"] /
    df["media_importe_3m_lag"].replace(0, np.nan)
)

df["subida_factura_flag_lag"] = np.where(
    df["subida_factura_lag"].fillna(0) > 0,
    1,
    0
)


# ======================================================
# 5. ELIMINAR VARIABLES CON LEAKAGE / NO VÁLIDAS
# ======================================================

quitar = [
    "cliente_id",
    "fecha",
    "churn",
    "alerta_churn",
    "zona_id",
    "zona_id_x",
    "zona_id_y",

    # variables del mismo mes, para evitar crítica de leakage temporal
    "importe_total",
    "dias_retraso_pago",
    "impago_flag"
]

df_model = df.drop(
    columns=[c for c in quitar if c in df.columns],
    errors="ignore"
)


# ======================================================
# 6. FEATURES Y TARGET
# ======================================================

X = df_model.drop(columns="churn_t_plus_1")
y = df_model["churn_t_plus_1"]

X = pd.get_dummies(X, drop_first=True)

X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median(numeric_only=True))
X = X.fillna(0)


# ======================================================
# 7. SPLIT TEMPORAL TRAIN / VALID / TEST
# ======================================================

fechas = df.loc[X.index, "fecha"]

datos = X.copy()
datos["target"] = y.values
datos["fecha"] = fechas.values

datos = datos.sort_values("fecha")

n = len(datos)

train_end = int(n * 0.70)
valid_end = int(n * 0.85)

train = datos.iloc[:train_end]
valid = datos.iloc[train_end:valid_end]
test = datos.iloc[valid_end:]

X_train = train.drop(columns=["target", "fecha"])
y_train = train["target"]

X_valid = valid.drop(columns=["target", "fecha"])
y_valid = valid["target"]

X_test = test.drop(columns=["target", "fecha"])
y_test = test["target"]

print("\nTamaños:")
print("Train:", X_train.shape)
print("Valid:", X_valid.shape)
print("Test:", X_test.shape)

print("\nChurn rate:")
print("Train:", round(y_train.mean() * 100, 4), "%")
print("Valid:", round(y_valid.mean() * 100, 4), "%")
print("Test:", round(y_test.mean() * 100, 4), "%")


# ======================================================
# 8. PESO CLASE MINORITARIA
# ======================================================

peso = (y_train == 0).sum() / (y_train == 1).sum()

print("\nscale_pos_weight base:", round(peso, 2))
print("scale_pos_weight usado:", round(peso * 2.5, 2))


# ======================================================
# 9. MODELO XGBOOST RECALL ALTO
# ======================================================

modelo = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.02,
    max_depth=3,
    min_child_weight=5,
    gamma=0.5,
    subsample=0.90,
    colsample_bytree=0.90,
    reg_alpha=1,
    reg_lambda=3,
    scale_pos_weight=peso * 2.5,
    objective="binary:logistic",
    eval_metric="aucpr",
    random_state=42,
    n_jobs=-1
)

modelo.fit(
    X_train,
    y_train,
    eval_set=[(X_valid, y_valid)],
    verbose=False
)


# ======================================================
# 10. THRESHOLD EN VALIDACIÓN
# ======================================================

proba_valid = modelo.predict_proba(X_valid)[:, 1]

resultados = []

for threshold in np.arange(0.10, 0.81, 0.01):

    pred_valid = (proba_valid >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_valid, pred_valid).ravel()

    resultados.append({
        "threshold": round(threshold, 2),
        "accuracy": accuracy_score(y_valid, pred_valid),
        "precision": precision_score(y_valid, pred_valid, zero_division=0),
        "recall": recall_score(y_valid, pred_valid, zero_division=0),
        "f1": f1_score(y_valid, pred_valid, zero_division=0),
        "f2": fbeta_score(y_valid, pred_valid, beta=2, zero_division=0),
        "clientes_churn_detectados": tp,
        "clientes_churn_no_detectados": fn,
        "falsos_positivos": fp,
        "total_predichos_churn": tp + fp
    })

df_thresholds = pd.DataFrame(resultados)

print("\nTHRESHOLDS ORDENADOS POR F2")
print(
    df_thresholds
    .sort_values(by="f2", ascending=False)
    .head(20)
)


# ======================================================
# 11. ELEGIR THRESHOLD FINAL
# ======================================================

recall_minimo = 0.35

candidatos = df_thresholds[df_thresholds["recall"] >= recall_minimo]

if len(candidatos) > 0:
    threshold_final = (
        candidatos
        .sort_values(by="f2", ascending=False)
        .iloc[0]["threshold"]
    )
else:
    threshold_final = (
        df_thresholds
        .sort_values(by="f2", ascending=False)
        .iloc[0]["threshold"]
    )

print("\nThreshold elegido en validación:", threshold_final)


# ======================================================
# 12. EVALUACIÓN FINAL EN TEST
# ======================================================

proba_test = modelo.predict_proba(X_test)[:, 1]
pred_test = (proba_test >= threshold_final).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test, pred_test).ravel()

accuracy = accuracy_score(y_test, pred_test)
precision = precision_score(y_test, pred_test, zero_division=0)
recall = recall_score(y_test, pred_test, zero_division=0)
f1 = f1_score(y_test, pred_test, zero_division=0)
f2 = fbeta_score(y_test, pred_test, beta=2, zero_division=0)
roc = roc_auc_score(y_test, proba_test)
pr_auc = average_precision_score(y_test, proba_test)

print("\n" + "="*70)
print("RESULTADO FINAL XGBOOST — TEST TEMPORAL")
print("="*70)

print("Threshold utilizado:", threshold_final)
print("Accuracy:", round(accuracy * 100, 2))
print("Precision churn:", round(precision, 4))
print("Recall churn:", round(recall, 4))
print("F1 churn:", round(f1, 4))
print("F2 churn:", round(f2, 4))
print("ROC-AUC:", round(roc, 4))
print("PR-AUC:", round(pr_auc, 4))

print("\nMatriz de confusión:")
print(confusion_matrix(y_test, pred_test))

print("\nClientes churn reales:", tp + fn)
print("Clientes churn detectados:", tp)
print("Clientes churn no detectados:", fn)
print("Falsos positivos:", fp)
print("Total predichos como churn:", tp + fp)

print("\nClassification Report:")
print(classification_report(y_test, pred_test, zero_division=0))


# ======================================================
# 13. IMPORTANCIA DE VARIABLES
# ======================================================

importancias = pd.DataFrame({
    "Variable": X_train.columns,
    "Importancia": modelo.feature_importances_
}).sort_values(
    by="Importancia",
    ascending=False
)

print("\nTOP 20 VARIABLES MÁS IMPORTANTES:")
print(importancias.head(20))

Shape inicial: (317579, 35)
Columnas: ['cliente_id', 'fecha', 'zona_id_x', 'tipo_plan_x', 'num_lineas_x', 'cargo_base', 'consumo_extra', 'descuento_aplicado', 'importe_total', 'dias_retraso_pago', 'impago_flag', 'variacion_consumo_pct', 'stress_calidad_lag', 'incidencia_masiva_lag', 'churn', 'zona_id_y', 'region', 'tipo_zona', 'poblacion_zona', 'edad', 'sexo', 'estado_civil', 'num_lineas_y', 'tipo_plan_y', 'tipo_dispositivo', 'ingreso_estimado', 'antiguedad_meses', 'descuento_activo', 'factura_mes_anterior', 'subida_factura', 'consumo_mes_anterior', 'cambio_consumo', 'impago_mes_anterior', 'retraso_mes_anterior', 'alerta_churn']

Tamaños:
Train: (215305, 54)
Valid: (46137, 54)
Test: (46137, 54)

Churn rate:
Train: 0.6818 %
Valid: 0.531 %
Test: 0.5137 %

scale_pos_weight base: 145.67
scale_pos_weight usado: 364.16

THRESHOLDS ORDENADOS POR F2
    threshold  accuracy  precision    recall        f1        f2  \
70       0.80  0.950192   0.017395  0.151020  0.031197  0.059543   
67       0

In [11]:
# ============================================================
# MODELO FINAL MEJORADO — XGBOOST CHURN T+1
# Split temporal + más features + búsqueda simple + threshold F2/Recall
# ============================================================

import pandas as pd
import numpy as np

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    fbeta_score, roc_auc_score, average_precision_score,
    confusion_matrix, classification_report
)

from xgboost import XGBClassifier


# ======================================================
# 1. CARGAR DATASET
# ======================================================

df = pd.read_csv("dataset_limpio_churn.csv")

print("Shape inicial:", df.shape)
print("Columnas iniciales:", df.columns.tolist())

df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")
df = df.sort_values(["cliente_id", "fecha"]).reset_index(drop=True)


# ======================================================
# 2. CREAR TARGET FUTURO: churn del mes siguiente
# ======================================================

df["churn_t_plus_1"] = (
    df.groupby("cliente_id")["churn"]
    .shift(-1)
)

df = df.dropna(subset=["churn_t_plus_1"])
df["churn_t_plus_1"] = df["churn_t_plus_1"].astype(int)


# ======================================================
# 3. FUNCIONES AUXILIARES
# ======================================================

def existe(col):
    return col in df.columns

def crear_lags(col, lags=[1, 2, 3]):
    if existe(col):
        for lag in lags:
            df[f"{col}_lag{lag}"] = df.groupby("cliente_id")[col].shift(lag)

def crear_rollings(col, ventanas=[3, 6]):
    if existe(col):
        for v in ventanas:
            df[f"{col}_media_{v}m_lag"] = (
                df.groupby("cliente_id")[col]
                .transform(lambda x: x.shift(1).rolling(v, min_periods=1).mean())
            )
            df[f"{col}_std_{v}m_lag"] = (
                df.groupby("cliente_id")[col]
                .transform(lambda x: x.shift(1).rolling(v, min_periods=2).std())
            )
            df[f"{col}_max_{v}m_lag"] = (
                df.groupby("cliente_id")[col]
                .transform(lambda x: x.shift(1).rolling(v, min_periods=1).max())
            )
            df[f"{col}_min_{v}m_lag"] = (
                df.groupby("cliente_id")[col]
                .transform(lambda x: x.shift(1).rolling(v, min_periods=1).min())
            )

def ratio_seguro(a, b):
    return a / b.replace(0, np.nan)


# ======================================================
# 4. VARIABLES HISTÓRICAS SIN LEAKAGE
# ======================================================

cols_historicas = [
    "importe_total",
    "dias_retraso_pago",
    "impago_flag",
    "consumo_datos_gb",
    "consumo_extra",
    "cambio_consumo",
    "stress_calidad_lag",
    "incidencia_masiva_lag",
    "variacion_consumo_pct"
]

for col in cols_historicas:
    crear_lags(col, lags=[1, 2, 3])
    crear_rollings(col, ventanas=[3, 6])


# ======================================================
# 5. FEATURES DERIVADAS IMPORTANTES
# ======================================================

if existe("dias_retraso_pago_lag1"):
    df["cliente_moroso_lag1"] = np.where(df["dias_retraso_pago_lag1"].fillna(0) > 0, 1, 0)

if existe("impago_flag_lag1") and existe("dias_retraso_pago_lag1"):
    df["riesgo_financiero_lag1"] = (
        df["impago_flag_lag1"].fillna(0) * 5
        + df["dias_retraso_pago_lag1"].fillna(0)
    )

if existe("importe_total_lag1") and existe("importe_total_media_3m_lag"):
    df["subida_factura_lag"] = (
        df["importe_total_lag1"] - df["importe_total_media_3m_lag"]
    )

    df["variacion_importe_lag"] = ratio_seguro(
        df["subida_factura_lag"],
        df["importe_total_media_3m_lag"]
    )

    df["subida_factura_flag_lag"] = np.where(
        df["subida_factura_lag"].fillna(0) > 0,
        1,
        0
    )

if existe("importe_total_media_3m_lag") and existe("importe_total_media_6m_lag"):
    df["tendencia_importe_3m_vs_6m"] = (
        df["importe_total_media_3m_lag"] - df["importe_total_media_6m_lag"]
    )

if existe("dias_retraso_pago_media_3m_lag") and existe("dias_retraso_pago_media_6m_lag"):
    df["tendencia_retraso_3m_vs_6m"] = (
        df["dias_retraso_pago_media_3m_lag"] - df["dias_retraso_pago_media_6m_lag"]
    )

if existe("impago_flag_media_3m_lag") and existe("impago_flag_media_6m_lag"):
    df["tendencia_impagos_3m_vs_6m"] = (
        df["impago_flag_media_3m_lag"] - df["impago_flag_media_6m_lag"]
    )

if existe("consumo_datos_gb_media_3m_lag") and existe("consumo_datos_gb_media_6m_lag"):
    df["tendencia_consumo_3m_vs_6m"] = (
        df["consumo_datos_gb_media_3m_lag"] - df["consumo_datos_gb_media_6m_lag"]
    )

if existe("stress_calidad_lag_media_3m_lag") and existe("stress_calidad_lag_media_6m_lag"):
    df["tendencia_stress_3m_vs_6m"] = (
        df["stress_calidad_lag_media_3m_lag"] - df["stress_calidad_lag_media_6m_lag"]
    )


# ======================================================
# 6. QUITAR VARIABLES CON LEAKAGE / IDs / NO VÁLIDAS
# ======================================================

quitar = [
    "cliente_id",
    "fecha",
    "churn",
    "alerta_churn",
    "zona_id",
    "zona_id_x",
    "zona_id_y",

    # variables del mismo mes
    "importe_total",
    "dias_retraso_pago",
    "impago_flag",
    "consumo_datos_gb"
]

df_model = df.drop(
    columns=[c for c in quitar if c in df.columns],
    errors="ignore"
)


# ======================================================
# 7. FEATURES Y TARGET
# ======================================================

X = df_model.drop(columns=["churn_t_plus_1"])
y = df_model["churn_t_plus_1"]

X = pd.get_dummies(X, drop_first=True)

X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median(numeric_only=True))
X = X.fillna(0)


# ======================================================
# 8. SPLIT TEMPORAL TRAIN / VALID / TEST
# ======================================================

fechas = df.loc[X.index, "fecha"]

datos = X.copy()
datos["target"] = y.values
datos["fecha"] = fechas.values

datos = datos.sort_values("fecha").reset_index(drop=True)

n = len(datos)

train_end = int(n * 0.70)
valid_end = int(n * 0.85)

train = datos.iloc[:train_end]
valid = datos.iloc[train_end:valid_end]
test = datos.iloc[valid_end:]

X_train = train.drop(columns=["target", "fecha"])
y_train = train["target"]

X_valid = valid.drop(columns=["target", "fecha"])
y_valid = valid["target"]

X_test = test.drop(columns=["target", "fecha"])
y_test = test["target"]

print("\nTamaños:")
print("Train:", X_train.shape)
print("Valid:", X_valid.shape)
print("Test:", X_test.shape)

print("\nChurn rate:")
print("Train:", round(y_train.mean() * 100, 4), "%")
print("Valid:", round(y_valid.mean() * 100, 4), "%")
print("Test:", round(y_test.mean() * 100, 4), "%")


# ======================================================
# 9. FUNCIÓN PARA BUSCAR THRESHOLD
# ======================================================

def buscar_threshold(y_true, proba, recall_minimo=0.45):

    resultados = []

    for threshold in np.arange(0.05, 0.91, 0.01):
        pred = (proba >= threshold).astype(int)

        tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()

        resultados.append({
            "threshold": round(threshold, 2),
            "accuracy": accuracy_score(y_true, pred),
            "precision": precision_score(y_true, pred, zero_division=0),
            "recall": recall_score(y_true, pred, zero_division=0),
            "f1": f1_score(y_true, pred, zero_division=0),
            "f2": fbeta_score(y_true, pred, beta=2, zero_division=0),
            "tp": tp,
            "fn": fn,
            "fp": fp,
            "total_predichos_churn": tp + fp
        })

    df_thr = pd.DataFrame(resultados)

    candidatos = df_thr[df_thr["recall"] >= recall_minimo]

    if len(candidatos) > 0:
        mejor = candidatos.sort_values("f2", ascending=False).iloc[0]
    else:
        mejor = df_thr.sort_values("f2", ascending=False).iloc[0]

    return df_thr, mejor


# ======================================================
# 10. COMPARAR VARIOS XGBOOST
# ======================================================

peso = (y_train == 0).sum() / (y_train == 1).sum()

print("\nscale_pos_weight base:", round(peso, 2))

configs = [
    {
        "nombre": "XGB_BALANCEADO",
        "scale": 2.5,
        "max_depth": 3,
        "min_child_weight": 5,
        "gamma": 0.5,
        "reg_alpha": 1,
        "reg_lambda": 3,
        "learning_rate": 0.02,
        "n_estimators": 1000
    },
    {
        "nombre": "XGB_RECALL_ALTO",
        "scale": 4.0,
        "max_depth": 2,
        "min_child_weight": 10,
        "gamma": 1,
        "reg_alpha": 3,
        "reg_lambda": 8,
        "learning_rate": 0.015,
        "n_estimators": 1500
    },
    {
        "nombre": "XGB_MUY_RECALL",
        "scale": 6.0,
        "max_depth": 2,
        "min_child_weight": 15,
        "gamma": 1.5,
        "reg_alpha": 5,
        "reg_lambda": 12,
        "learning_rate": 0.01,
        "n_estimators": 2000
    },
    {
        "nombre": "XGB_MENOS_REGULARIZADO",
        "scale": 3.0,
        "max_depth": 4,
        "min_child_weight": 5,
        "gamma": 0.5,
        "reg_alpha": 1,
        "reg_lambda": 5,
        "learning_rate": 0.02,
        "n_estimators": 1200
    }
]

resumen_modelos = []
modelos_entrenados = {}

for cfg in configs:

    print("\nEntrenando:", cfg["nombre"])

    modelo = XGBClassifier(
        n_estimators=cfg["n_estimators"],
        learning_rate=cfg["learning_rate"],
        max_depth=cfg["max_depth"],
        min_child_weight=cfg["min_child_weight"],
        gamma=cfg["gamma"],
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=cfg["reg_alpha"],
        reg_lambda=cfg["reg_lambda"],
        scale_pos_weight=peso * cfg["scale"],
        objective="binary:logistic",
        eval_metric="aucpr",
        random_state=42,
        n_jobs=-1
    )

    modelo.fit(
        X_train,
        y_train,
        eval_set=[(X_valid, y_valid)],
        verbose=False
    )

    proba_valid = modelo.predict_proba(X_valid)[:, 1]

    df_thr, mejor_thr = buscar_threshold(
        y_valid,
        proba_valid,
        recall_minimo=0.45
    )

    resumen_modelos.append({
        "modelo": cfg["nombre"],
        "threshold": mejor_thr["threshold"],
        "valid_accuracy": mejor_thr["accuracy"],
        "valid_precision": mejor_thr["precision"],
        "valid_recall": mejor_thr["recall"],
        "valid_f1": mejor_thr["f1"],
        "valid_f2": mejor_thr["f2"],
        "valid_tp": mejor_thr["tp"],
        "valid_fn": mejor_thr["fn"],
        "valid_fp": mejor_thr["fp"],
        "valid_total_pred_churn": mejor_thr["total_predichos_churn"],
        "valid_roc_auc": roc_auc_score(y_valid, proba_valid),
        "valid_pr_auc": average_precision_score(y_valid, proba_valid)
    })

    modelos_entrenados[cfg["nombre"]] = {
        "modelo": modelo,
        "thresholds": df_thr,
        "mejor_threshold": mejor_thr
    }


resumen = pd.DataFrame(resumen_modelos)

print("\n" + "="*80)
print("RESUMEN VALIDACIÓN")
print("="*80)

print(
    resumen.sort_values(
        by=["valid_f2", "valid_recall", "valid_pr_auc"],
        ascending=False
    )
)


# ======================================================
# 11. ELEGIR MEJOR MODELO POR F2 EN VALIDACIÓN
# ======================================================

mejor_nombre = (
    resumen.sort_values(
        by=["valid_f2", "valid_recall", "valid_pr_auc"],
        ascending=False
    )
    .iloc[0]["modelo"]
)

mejor_modelo = modelos_entrenados[mejor_nombre]["modelo"]
threshold_final = modelos_entrenados[mejor_nombre]["mejor_threshold"]["threshold"]

print("\nMejor modelo elegido:", mejor_nombre)
print("Threshold final elegido:", threshold_final)


# ======================================================
# 12. EVALUACIÓN FINAL EN TEST
# ======================================================

proba_test = mejor_modelo.predict_proba(X_test)[:, 1]
pred_test = (proba_test >= threshold_final).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test, pred_test).ravel()

accuracy = accuracy_score(y_test, pred_test)
precision = precision_score(y_test, pred_test, zero_division=0)
recall = recall_score(y_test, pred_test, zero_division=0)
f1 = f1_score(y_test, pred_test, zero_division=0)
f2 = fbeta_score(y_test, pred_test, beta=2, zero_division=0)
roc = roc_auc_score(y_test, proba_test)
pr_auc = average_precision_score(y_test, proba_test)

print("\n" + "="*80)
print("RESULTADO FINAL — MEJOR XGBOOST EN TEST TEMPORAL")
print("="*80)

print("Modelo:", mejor_nombre)
print("Threshold utilizado:", threshold_final)

print("Accuracy:", round(accuracy * 100, 2))
print("Precision churn:", round(precision, 4))
print("Recall churn:", round(recall, 4))
print("F1 churn:", round(f1, 4))
print("F2 churn:", round(f2, 4))
print("ROC-AUC:", round(roc, 4))
print("PR-AUC:", round(pr_auc, 4))

print("\nMatriz de confusión:")
print(confusion_matrix(y_test, pred_test))

print("\nClientes churn reales:", tp + fn)
print("Clientes churn detectados:", tp)
print("Clientes churn no detectados:", fn)
print("Falsos positivos:", fp)
print("Total predichos como churn:", tp + fp)

print("\nClassification Report:")
print(classification_report(y_test, pred_test, zero_division=0))


# ======================================================
# 13. TOP VARIABLES IMPORTANTES
# ======================================================

importancias = pd.DataFrame({
    "Variable": X_train.columns,
    "Importancia": mejor_modelo.feature_importances_
}).sort_values(
    by="Importancia",
    ascending=False
)

print("\nTOP 30 VARIABLES MÁS IMPORTANTES:")
print(importancias.head(30))


# ======================================================
# 14. TABLA DE THRESHOLDS DEL MEJOR MODELO
# ======================================================

print("\nTOP 20 THRESHOLDS DEL MEJOR MODELO POR F2:")
print(
    modelos_entrenados[mejor_nombre]["thresholds"]
    .sort_values("f2", ascending=False)
    .head(20)
)

Shape inicial: (317579, 35)
Columnas iniciales: ['cliente_id', 'fecha', 'zona_id_x', 'tipo_plan_x', 'num_lineas_x', 'cargo_base', 'consumo_extra', 'descuento_aplicado', 'importe_total', 'dias_retraso_pago', 'impago_flag', 'variacion_consumo_pct', 'stress_calidad_lag', 'incidencia_masiva_lag', 'churn', 'zona_id_y', 'region', 'tipo_zona', 'poblacion_zona', 'edad', 'sexo', 'estado_civil', 'num_lineas_y', 'tipo_plan_y', 'tipo_dispositivo', 'ingreso_estimado', 'antiguedad_meses', 'descuento_activo', 'factura_mes_anterior', 'subida_factura', 'consumo_mes_anterior', 'cambio_consumo', 'impago_mes_anterior', 'retraso_mes_anterior', 'alerta_churn']


/var/folders/fz/t811yjh570727qqxzhgpjjgc0000gn/T/ipykernel_38957/2119735614.py:63: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_std_{v}m_lag"] = (
/var/folders/fz/t811yjh570727qqxzhgpjjgc0000gn/T/ipykernel_38957/2119735614.py:67: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_max_{v}m_lag"] = (
/var/folders/fz/t811yjh570727qqxzhgpjjgc0000gn/T/ipykernel_38957/2119735614.py:71: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor 


Tamaños:
Train: (215305, 137)
Valid: (46137, 137)
Test: (46137, 137)

Churn rate:
Train: 0.6818 %
Valid: 0.531 %
Test: 0.5137 %

scale_pos_weight base: 145.67

Entrenando: XGB_BALANCEADO

Entrenando: XGB_RECALL_ALTO

Entrenando: XGB_MUY_RECALL

Entrenando: XGB_MENOS_REGULARIZADO

RESUMEN VALIDACIÓN
                   modelo  threshold  valid_accuracy  valid_precision  \
1         XGB_RECALL_ALTO       0.76        0.768841         0.010614   
2          XGB_MUY_RECALL       0.82        0.758393         0.010419   
0          XGB_BALANCEADO       0.62        0.759434         0.010199   
3  XGB_MENOS_REGULARIZADO       0.51        0.718404         0.009089   

   valid_recall  valid_f1  valid_f2  valid_tp  valid_fn  valid_fp  \
1      0.461224  0.020751  0.048598     113.0     132.0   10533.0   
2      0.473469  0.020388  0.047878     116.0     129.0   11018.0   
0      0.461224  0.019956  0.046849     113.0     132.0   10967.0   
3      0.481633  0.017841  0.042255     118.0     127.0  

In [13]:
# ============================================================
# MODELO CHURN MEJORADO — XGBOOST + FEATURE SELECTION
# ============================================================

import pandas as pd
import numpy as np

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    fbeta_score, roc_auc_score, average_precision_score,
    confusion_matrix, classification_report
)

from xgboost import XGBClassifier


# ======================================================
# 1. CARGAR DATASET
# ======================================================

df = pd.read_csv("dataset_limpio_churn.csv")

df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")
df = df.sort_values(["cliente_id", "fecha"]).reset_index(drop=True)

print("Shape inicial:", df.shape)


# ======================================================
# 2. TARGET FUTURO
# ======================================================

df["churn_t_plus_1"] = df.groupby("cliente_id")["churn"].shift(-1)

df = df.dropna(subset=["churn_t_plus_1"])
df["churn_t_plus_1"] = df["churn_t_plus_1"].astype(int)


# ======================================================
# 3. CREAR VARIABLES HISTÓRICAS SIN LEAKAGE
# ======================================================

def crear_lags_y_rollings(df, col):
    if col not in df.columns:
        return df

    for lag in [1, 2, 3]:
        df[f"{col}_lag{lag}"] = df.groupby("cliente_id")[col].shift(lag)

    for ventana in [3, 6]:
        df[f"{col}_mean_{ventana}m_lag"] = (
            df.groupby("cliente_id")[col]
            .transform(lambda x: x.shift(1).rolling(ventana, min_periods=1).mean())
        )

        df[f"{col}_std_{ventana}m_lag"] = (
            df.groupby("cliente_id")[col]
            .transform(lambda x: x.shift(1).rolling(ventana, min_periods=2).std())
        )

        df[f"{col}_max_{ventana}m_lag"] = (
            df.groupby("cliente_id")[col]
            .transform(lambda x: x.shift(1).rolling(ventana, min_periods=1).max())
        )

    return df


variables_historicas = [
    "importe_total",
    "dias_retraso_pago",
    "impago_flag",
    "consumo_extra",
    "variacion_consumo_pct",
    "stress_calidad_lag",
    "incidencia_masiva_lag",
    "factura_mes_anterior",
    "consumo_mes_anterior",
    "cambio_consumo",
    "impago_mes_anterior",
    "retraso_mes_anterior"
]

for col in variables_historicas:
    df = crear_lags_y_rollings(df, col)


# ======================================================
# 4. VARIABLES DERIVADAS
# ======================================================

if "dias_retraso_pago_lag1" in df.columns:
    df["cliente_moroso_lag1"] = np.where(
        df["dias_retraso_pago_lag1"].fillna(0) > 0,
        1,
        0
    )

if "impago_flag_lag1" in df.columns and "dias_retraso_pago_lag1" in df.columns:
    df["riesgo_financiero_lag1"] = (
        df["impago_flag_lag1"].fillna(0) * 5
        + df["dias_retraso_pago_lag1"].fillna(0)
    )

if "importe_total_lag1" in df.columns and "importe_total_mean_3m_lag" in df.columns:
    df["subida_factura_lag"] = (
        df["importe_total_lag1"] - df["importe_total_mean_3m_lag"]
    )

    df["ratio_factura_lag"] = (
        df["importe_total_lag1"] /
        df["importe_total_mean_3m_lag"].replace(0, np.nan)
    )

if "dias_retraso_pago_lag1" in df.columns and "dias_retraso_pago_lag3" in df.columns:
    df["deterioro_retraso"] = (
        df["dias_retraso_pago_lag1"] - df["dias_retraso_pago_lag3"]
    )

if "impago_flag_lag1" in df.columns and "impago_flag_lag2" in df.columns and "impago_flag_lag3" in df.columns:
    df["racha_impagos_3m"] = (
        df["impago_flag_lag1"].fillna(0)
        + df["impago_flag_lag2"].fillna(0)
        + df["impago_flag_lag3"].fillna(0)
    )


# ======================================================
# 5. QUITAR LEAKAGE
# ======================================================

quitar = [
    "cliente_id",
    "fecha",
    "churn",
    "alerta_churn",
    "zona_id",
    "zona_id_x",
    "zona_id_y",

    # variables del mismo mes
    "importe_total",
    "dias_retraso_pago",
    "impago_flag"
]

df_model = df.drop(columns=[c for c in quitar if c in df.columns], errors="ignore")


# ======================================================
# 6. X / y
# ======================================================

X = df_model.drop(columns=["churn_t_plus_1"])
y = df_model["churn_t_plus_1"]

X = pd.get_dummies(X, drop_first=True)

X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median(numeric_only=True))
X = X.fillna(0)


# ======================================================
# 7. SPLIT TEMPORAL
# ======================================================

fechas = df.loc[X.index, "fecha"]

datos = X.copy()
datos["target"] = y.values
datos["fecha"] = fechas.values

datos = datos.sort_values("fecha").reset_index(drop=True)

n = len(datos)

train_end = int(n * 0.70)
valid_end = int(n * 0.85)

train = datos.iloc[:train_end]
valid = datos.iloc[train_end:valid_end]
test = datos.iloc[valid_end:]

X_train = train.drop(columns=["target", "fecha"])
y_train = train["target"]

X_valid = valid.drop(columns=["target", "fecha"])
y_valid = valid["target"]

X_test = test.drop(columns=["target", "fecha"])
y_test = test["target"]

print("\nTamaños:")
print("Train:", X_train.shape)
print("Valid:", X_valid.shape)
print("Test:", X_test.shape)

print("\nChurn rate:")
print("Train:", round(y_train.mean() * 100, 4), "%")
print("Valid:", round(y_valid.mean() * 100, 4), "%")
print("Test:", round(y_test.mean() * 100, 4), "%")


# ======================================================
# 8. MODELO BASE PARA SACAR IMPORTANCIAS
# ======================================================

peso = (y_train == 0).sum() / (y_train == 1).sum()

modelo_base = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.02,
    max_depth=3,
    min_child_weight=8,
    gamma=1,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=2,
    reg_lambda=8,
    scale_pos_weight=peso * 3,
    objective="binary:logistic",
    eval_metric="aucpr",
    random_state=42,
    n_jobs=-1
)

modelo_base.fit(
    X_train,
    y_train,
    eval_set=[(X_valid, y_valid)],
    verbose=False
)


# ======================================================
# 9. SELECCIÓN DE TOP VARIABLES
# ======================================================

importancias = pd.DataFrame({
    "Variable": X_train.columns,
    "Importancia": modelo_base.feature_importances_
}).sort_values("Importancia", ascending=False)

print("\nTOP 40 VARIABLES:")
print(importancias.head(40))

top_n = 40

features_seleccionadas = importancias.head(top_n)["Variable"].tolist()

X_train_sel = X_train[features_seleccionadas]
X_valid_sel = X_valid[features_seleccionadas]
X_test_sel = X_test[features_seleccionadas]

print("\nNúmero de variables seleccionadas:", len(features_seleccionadas))


# ======================================================
# 10. MODELO FINAL CON VARIABLES SELECCIONADAS
# ======================================================

modelo_final = XGBClassifier(
    n_estimators=1500,
    learning_rate=0.015,
    max_depth=3,
    min_child_weight=10,
    gamma=1,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=3,
    reg_lambda=10,
    scale_pos_weight=peso * 3.5,
    objective="binary:logistic",
    eval_metric="aucpr",
    random_state=42,
    n_jobs=-1
)

modelo_final.fit(
    X_train_sel,
    y_train,
    eval_set=[(X_valid_sel, y_valid)],
    verbose=False
)


# ======================================================
# 11. BUSCAR THRESHOLD
# ======================================================

def evaluar_thresholds(y_true, proba):

    resultados = []

    for threshold in np.arange(0.10, 0.91, 0.01):

        pred = (proba >= threshold).astype(int)

        tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()

        resultados.append({
            "threshold": round(threshold, 2),
            "accuracy": accuracy_score(y_true, pred),
            "precision": precision_score(y_true, pred, zero_division=0),
            "recall": recall_score(y_true, pred, zero_division=0),
            "f1": f1_score(y_true, pred, zero_division=0),
            "f2": fbeta_score(y_true, pred, beta=2, zero_division=0),
            "tp": tp,
            "fn": fn,
            "fp": fp,
            "total_predichos_churn": tp + fp
        })

    return pd.DataFrame(resultados)


proba_valid = modelo_final.predict_proba(X_valid_sel)[:, 1]

df_thresholds = evaluar_thresholds(y_valid, proba_valid)

print("\nTHRESHOLDS ORDENADOS POR F2")
print(
    df_thresholds
    .sort_values("f2", ascending=False)
    .head(20)
)

print("\nTHRESHOLDS ORDENADOS POR RECALL")
print(
    df_thresholds
    .sort_values("recall", ascending=False)
    .head(20)
)


# ======================================================
# 12. ELEGIR THRESHOLD FINAL
# ======================================================

recall_minimo = 0.40

candidatos = df_thresholds[df_thresholds["recall"] >= recall_minimo]

if len(candidatos) > 0:
    threshold_final = (
        candidatos
        .sort_values("f2", ascending=False)
        .iloc[0]["threshold"]
    )
else:
    threshold_final = (
        df_thresholds
        .sort_values("f2", ascending=False)
        .iloc[0]["threshold"]
    )

print("\nThreshold elegido:", threshold_final)


# ======================================================
# 13. TEST FINAL
# ======================================================

proba_test = modelo_final.predict_proba(X_test_sel)[:, 1]
pred_test = (proba_test >= threshold_final).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test, pred_test).ravel()

accuracy = accuracy_score(y_test, pred_test)
precision = precision_score(y_test, pred_test, zero_division=0)
recall = recall_score(y_test, pred_test, zero_division=0)
f1 = f1_score(y_test, pred_test, zero_division=0)
f2 = fbeta_score(y_test, pred_test, beta=2, zero_division=0)
roc = roc_auc_score(y_test, proba_test)
pr_auc = average_precision_score(y_test, proba_test)

print("\n" + "="*80)
print("RESULTADO FINAL — XGBOOST FEATURE SELECTION")
print("="*80)

print("Threshold utilizado:", threshold_final)
print("Accuracy:", round(accuracy * 100, 2))
print("Precision churn:", round(precision, 4))
print("Recall churn:", round(recall, 4))
print("F1 churn:", round(f1, 4))
print("F2 churn:", round(f2, 4))
print("ROC-AUC:", round(roc, 4))
print("PR-AUC:", round(pr_auc, 4))

print("\nMatriz de confusión:")
print(confusion_matrix(y_test, pred_test))

print("\nClientes churn reales:", tp + fn)
print("Clientes churn detectados:", tp)
print("Clientes churn no detectados:", fn)
print("Falsos positivos:", fp)
print("Total predichos como churn:", tp + fp)

print("\nClassification Report:")
print(classification_report(y_test, pred_test, zero_division=0))


# ======================================================
# 14. VARIABLES FINALES
# ======================================================

importancias_finales = pd.DataFrame({
    "Variable": X_train_sel.columns,
    "Importancia": modelo_final.feature_importances_
}).sort_values("Importancia", ascending=False)

print("\nTOP VARIABLES DEL MODELO FINAL:")
print(importancias_finales.head(40))

Shape inicial: (317579, 35)


/var/folders/fz/t811yjh570727qqxzhgpjjgc0000gn/T/ipykernel_38957/3120342257.py:48: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_lag{lag}"] = df.groupby("cliente_id")[col].shift(lag)
/var/folders/fz/t811yjh570727qqxzhgpjjgc0000gn/T/ipykernel_38957/3120342257.py:51: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_mean_{ventana}m_lag"] = (
/var/folders/fz/t811yjh570727qqxzhgpjjgc0000gn/T/ipykernel_38957/3120342257.py:56: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling 


Tamaños:
Train: (215305, 154)
Valid: (46137, 154)
Test: (46137, 154)

Churn rate:
Train: 0.6818 %
Valid: 0.531 %
Test: 0.5137 %

TOP 40 VARIABLES:
                             Variable  Importancia
128            riesgo_financiero_lag1     0.073543
17                impago_mes_anterior     0.041280
18               retraso_mes_anterior     0.019638
28             dias_retraso_pago_lag1     0.015615
64            stress_calidad_lag_lag1     0.013191
65            stress_calidad_lag_lag2     0.012225
5                  stress_calidad_lag     0.011901
134               tipo_plan_x_Premium     0.011868
11                   antiguedad_meses     0.011555
10                   ingreso_estimado     0.011481
69      stress_calidad_lag_max_3m_lag     0.011119
72      stress_calidad_lag_max_6m_lag     0.008892
150               tipo_plan_y_Premium     0.008783
66            stress_calidad_lag_lag3     0.008593
140               tipo_zona_suburbana     0.008216
44             impago_flag_std_6m_la

In [16]:
# ============================================================
# MODELO FINAL TFM — XGBOOST CHURN T+1 DEFENDIBLE
# ============================================================

import pandas as pd
import numpy as np

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    fbeta_score, roc_auc_score, average_precision_score,
    confusion_matrix, classification_report
)

from xgboost import XGBClassifier


df = pd.read_csv("dataset_limpio_churn.csv")

df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")
df = df.sort_values(["cliente_id", "fecha"]).reset_index(drop=True)

print("Shape inicial:", df.shape)


# TARGET FUTURO
df["churn_t_plus_1"] = df.groupby("cliente_id")["churn"].shift(-1)
df = df.dropna(subset=["churn_t_plus_1"])
df["churn_t_plus_1"] = df["churn_t_plus_1"].astype(int)


# FEATURES HISTÓRICAS SIN LEAKAGE
def crear_lags_rollings(col):
    if col not in df.columns:
        return

    for lag in [1, 2, 3]:
        df[f"{col}_lag{lag}"] = df.groupby("cliente_id")[col].shift(lag)

    for v in [3, 6]:
        df[f"{col}_media_{v}m_lag"] = (
            df.groupby("cliente_id")[col]
            .transform(lambda x: x.shift(1).rolling(v, min_periods=1).mean())
        )

        df[f"{col}_std_{v}m_lag"] = (
            df.groupby("cliente_id")[col]
            .transform(lambda x: x.shift(1).rolling(v, min_periods=2).std())
        )

        df[f"{col}_max_{v}m_lag"] = (
            df.groupby("cliente_id")[col]
            .transform(lambda x: x.shift(1).rolling(v, min_periods=1).max())
        )

        df[f"{col}_min_{v}m_lag"] = (
            df.groupby("cliente_id")[col]
            .transform(lambda x: x.shift(1).rolling(v, min_periods=1).min())
        )


cols_historicas = [
    "importe_total",
    "dias_retraso_pago",
    "impago_flag",
    "consumo_extra",
    "variacion_consumo_pct",
    "stress_calidad_lag",
    "incidencia_masiva_lag",
    "factura_mes_anterior",
    "consumo_mes_anterior",
    "cambio_consumo",
    "impago_mes_anterior",
    "retraso_mes_anterior"
]

for col in cols_historicas:
    crear_lags_rollings(col)

df = df.copy()


# VARIABLES DERIVADAS
if "dias_retraso_pago_lag1" in df.columns:
    df["cliente_moroso_lag1"] = np.where(df["dias_retraso_pago_lag1"].fillna(0) > 0, 1, 0)

if "impago_flag_lag1" in df.columns and "dias_retraso_pago_lag1" in df.columns:
    df["riesgo_financiero_lag1"] = (
        df["impago_flag_lag1"].fillna(0) * 5
        + df["dias_retraso_pago_lag1"].fillna(0)
    )

if all(c in df.columns for c in ["impago_flag_lag1", "impago_flag_lag2", "impago_flag_lag3"]):
    df["racha_impagos_3m"] = (
        df["impago_flag_lag1"].fillna(0)
        + df["impago_flag_lag2"].fillna(0)
        + df["impago_flag_lag3"].fillna(0)
    )

if "dias_retraso_pago_lag1" in df.columns and "dias_retraso_pago_lag3" in df.columns:
    df["deterioro_retraso"] = df["dias_retraso_pago_lag1"] - df["dias_retraso_pago_lag3"]


# QUITAR LEAKAGE / IDS
quitar = [
    "cliente_id",
    "fecha",
    "churn",
    "alerta_churn",
    "zona_id",
    "zona_id_x",
    "zona_id_y",

    # variables del mismo mes
    "importe_total",
    "dias_retraso_pago",
    "impago_flag"
]

df_model = df.drop(columns=[c for c in quitar if c in df.columns], errors="ignore")


# X / y
X = df_model.drop(columns=["churn_t_plus_1"])
y = df_model["churn_t_plus_1"]

X = pd.get_dummies(X, drop_first=True)

X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median(numeric_only=True))
X = X.fillna(0)


# SPLIT TEMPORAL
fechas = df.loc[X.index, "fecha"]

datos = X.copy()
datos["target"] = y.values
datos["fecha"] = fechas.values

datos = datos.sort_values("fecha").reset_index(drop=True)

n = len(datos)
train_end = int(n * 0.70)
valid_end = int(n * 0.85)

train = datos.iloc[:train_end]
valid = datos.iloc[train_end:valid_end]
test = datos.iloc[valid_end:]

X_train = train.drop(columns=["target", "fecha"])
y_train = train["target"]

X_valid = valid.drop(columns=["target", "fecha"])
y_valid = valid["target"]

X_test = test.drop(columns=["target", "fecha"])
y_test = test["target"]

print("\nTamaños:")
print("Train:", X_train.shape)
print("Valid:", X_valid.shape)
print("Test:", X_test.shape)

print("\nChurn rate:")
print("Train:", round(y_train.mean() * 100, 4), "%")
print("Valid:", round(y_valid.mean() * 100, 4), "%")
print("Test:", round(y_test.mean() * 100, 4), "%")


# THRESHOLD FUNCTION
def buscar_threshold(y_true, proba, recall_minimo=0.40):
    resultados = []

    for threshold in np.arange(0.05, 0.91, 0.01):
        pred = (proba >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()

        resultados.append({
            "threshold": round(threshold, 2),
            "accuracy": accuracy_score(y_true, pred),
            "precision": precision_score(y_true, pred, zero_division=0),
            "recall": recall_score(y_true, pred, zero_division=0),
            "f1": f1_score(y_true, pred, zero_division=0),
            "f2": fbeta_score(y_true, pred, beta=2, zero_division=0),
            "tp": tp,
            "fn": fn,
            "fp": fp,
            "total_predichos_churn": tp + fp
        })

    df_thr = pd.DataFrame(resultados)

    candidatos = df_thr[df_thr["recall"] >= recall_minimo]

    if len(candidatos) > 0:
        mejor = candidatos.sort_values("f2", ascending=False).iloc[0]
    else:
        mejor = df_thr.sort_values("f2", ascending=False).iloc[0]

    return df_thr, mejor


# MODELOS XGBOOST
peso = (y_train == 0).sum() / (y_train == 1).sum()
print("\nscale_pos_weight base:", round(peso, 2))

configs = [
    {
        "nombre": "XGB_BALANCEADO",
        "scale": 2.5,
        "max_depth": 3,
        "min_child_weight": 5,
        "gamma": 0.5,
        "reg_alpha": 1,
        "reg_lambda": 3,
        "learning_rate": 0.02,
        "n_estimators": 1000
    },
    {
        "nombre": "XGB_RECALL_ALTO",
        "scale": 4.0,
        "max_depth": 2,
        "min_child_weight": 10,
        "gamma": 1,
        "reg_alpha": 3,
        "reg_lambda": 8,
        "learning_rate": 0.015,
        "n_estimators": 1500
    },
    {
        "nombre": "XGB_MUY_RECALL",
        "scale": 6.0,
        "max_depth": 2,
        "min_child_weight": 15,
        "gamma": 1.5,
        "reg_alpha": 5,
        "reg_lambda": 12,
        "learning_rate": 0.01,
        "n_estimators": 2000
    },
    {
        "nombre": "XGB_MENOS_REGULARIZADO",
        "scale": 3.0,
        "max_depth": 4,
        "min_child_weight": 5,
        "gamma": 0.5,
        "reg_alpha": 1,
        "reg_lambda": 5,
        "learning_rate": 0.02,
        "n_estimators": 1200
    }
]

resumen_modelos = []
modelos_entrenados = {}

for cfg in configs:

    print("\nEntrenando:", cfg["nombre"])

    modelo = XGBClassifier(
        n_estimators=cfg["n_estimators"],
        learning_rate=cfg["learning_rate"],
        max_depth=cfg["max_depth"],
        min_child_weight=cfg["min_child_weight"],
        gamma=cfg["gamma"],
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=cfg["reg_alpha"],
        reg_lambda=cfg["reg_lambda"],
        scale_pos_weight=peso * cfg["scale"],
        objective="binary:logistic",
        eval_metric="aucpr",
        random_state=42,
        n_jobs=-1
    )

    modelo.fit(
        X_train,
        y_train,
        eval_set=[(X_valid, y_valid)],
        verbose=False
    )

    proba_valid = modelo.predict_proba(X_valid)[:, 1]

    df_thr, mejor_thr = buscar_threshold(
        y_valid,
        proba_valid,
        recall_minimo=0.40
    )

    resumen_modelos.append({
        "modelo": cfg["nombre"],
        "threshold": mejor_thr["threshold"],
        "valid_accuracy": mejor_thr["accuracy"],
        "valid_precision": mejor_thr["precision"],
        "valid_recall": mejor_thr["recall"],
        "valid_f1": mejor_thr["f1"],
        "valid_f2": mejor_thr["f2"],
        "valid_tp": mejor_thr["tp"],
        "valid_fn": mejor_thr["fn"],
        "valid_fp": mejor_thr["fp"],
        "valid_pr_auc": average_precision_score(y_valid, proba_valid),
        "valid_roc_auc": roc_auc_score(y_valid, proba_valid)
    })

    modelos_entrenados[cfg["nombre"]] = {
        "modelo": modelo,
        "threshold": mejor_thr["threshold"],
        "thresholds": df_thr
    }


resumen = pd.DataFrame(resumen_modelos)

print("\n" + "="*80)
print("RESUMEN VALIDACIÓN")
print("="*80)

print(
    resumen.sort_values(
        by=["valid_f2", "valid_recall", "valid_pr_auc"],
        ascending=False
    )
)


# ELEGIR MEJOR MODELO EN VALIDACIÓN
mejor_nombre = (
    resumen.sort_values(
        by=["valid_f2", "valid_recall", "valid_pr_auc"],
        ascending=False
    )
    .iloc[0]["modelo"]
)

modelo_final = modelos_entrenados[mejor_nombre]["modelo"]
threshold_final = modelos_entrenados[mejor_nombre]["threshold"]

print("\nMejor modelo elegido:", mejor_nombre)
print("Threshold final elegido:", threshold_final)


# TEST FINAL
proba_test = modelo_final.predict_proba(X_test)[:, 1]
pred_test = (proba_test >= threshold_final).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test, pred_test).ravel()

accuracy = accuracy_score(y_test, pred_test)
precision = precision_score(y_test, pred_test, zero_division=0)
recall = recall_score(y_test, pred_test, zero_division=0)
f1 = f1_score(y_test, pred_test, zero_division=0)
f2 = fbeta_score(y_test, pred_test, beta=2, zero_division=0)
roc = roc_auc_score(y_test, proba_test)
pr_auc = average_precision_score(y_test, proba_test)

print("\n" + "="*80)
print("RESULTADO FINAL — XGBOOST CHURN T+1")
print("="*80)

print("Modelo:", mejor_nombre)
print("Threshold utilizado:", threshold_final)

print("Accuracy:", round(accuracy * 100, 2))
print("Precision churn:", round(precision, 4))
print("Recall churn:", round(recall, 4))
print("F1 churn:", round(f1, 4))
print("F2 churn:", round(f2, 4))
print("ROC-AUC:", round(roc, 4))
print("PR-AUC:", round(pr_auc, 4))

print("\nMatriz de confusión:")
print(confusion_matrix(y_test, pred_test))

print("\nClientes churn reales:", tp + fn)
print("Clientes churn detectados:", tp)
print("Clientes churn no detectados:", fn)
print("Falsos positivos:", fp)
print("Total predichos como churn:", tp + fp)

print("\nClassification Report:")
print(classification_report(y_test, pred_test, zero_division=0))


# IMPORTANCIAS
importancias = pd.DataFrame({
    "Variable": X_train.columns,
    "Importancia": modelo_final.feature_importances_
}).sort_values("Importancia", ascending=False)

print("\nTOP 30 VARIABLES MÁS IMPORTANTES:")
print(importancias.head(30))

print("\nTOP 20 THRESHOLDS DEL MEJOR MODELO:")
print(
    modelos_entrenados[mejor_nombre]["thresholds"]
    .sort_values("f2", ascending=False)
    .head(20)
)

Shape inicial: (317579, 35)


/var/folders/fz/t811yjh570727qqxzhgpjjgc0000gn/T/ipykernel_38957/379315576.py:45: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_std_{v}m_lag"] = (
/var/folders/fz/t811yjh570727qqxzhgpjjgc0000gn/T/ipykernel_38957/379315576.py:50: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_max_{v}m_lag"] = (
/var/folders/fz/t811yjh570727qqxzhgpjjgc0000gn/T/ipykernel_38957/379315576.py:55: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor per


Tamaños:
Train: (215305, 176)
Valid: (46137, 176)
Test: (46137, 176)

Churn rate:
Train: 0.6818 %
Valid: 0.531 %
Test: 0.5137 %

scale_pos_weight base: 145.67

Entrenando: XGB_BALANCEADO

Entrenando: XGB_RECALL_ALTO

Entrenando: XGB_MUY_RECALL

Entrenando: XGB_MENOS_REGULARIZADO

RESUMEN VALIDACIÓN
                   modelo  threshold  valid_accuracy  valid_precision  \
2          XGB_MUY_RECALL       0.85        0.826322         0.012550   
1         XGB_RECALL_ALTO       0.79        0.827969         0.012424   
0          XGB_BALANCEADO       0.65        0.802458         0.011242   
3  XGB_MENOS_REGULARIZADO       0.57        0.786635         0.010005   

   valid_recall  valid_f1  valid_f2  valid_tp  valid_fn  valid_fp  \
2      0.408163  0.024352  0.055878     100.0     145.0    7868.0   
1      0.400000  0.024099  0.055255      98.0     147.0    7790.0   
0      0.416327  0.021893  0.050731     102.0     143.0    8971.0   
3      0.400000  0.019522  0.045476      98.0     147.0  